# Outfit Recommendation


In [75]:
from langchain_community.tools.tavily_search import TavilySearchResults
from langgraph.prebuilt import create_react_agent
from langchain_groq import ChatGroq
from langchain.tools import Tool
from dotenv import load_dotenv
import os
import json
import re

load_dotenv()

class FashionSearchAgent:
    def __init__(self):
        # Load environment variables

        self.prompt_dari_text = """This tool only used for searching product where the input is event or occasion. Your task is to search for real, available product links for fashion items based on the user's event or occasion input.

            IMPORTANT SEARCH INSTRUCTIONS:
            1. For each category, first formulate a specific search query like "baju pria formal lebaran tokopedia" or "sepatu wanita casual shopee"
            2. YOU MUST RETURN DIRECT PRODUCT LINKS ONLY. Valid links should point to specific products
            3. DO NOT use "find" or "search" keywords links like "https://www.tokopedia.com/find/..." or "https://shopee.co.id/search?keyword=..." - these are NOT acceptable
            4. Click through to actual product pages to get the direct product URLs
            5. If you cannot find a suitable item after searching, remove that category from the JSON object. Do not include empty categories.
            6. You need to make sure the link is available, because i always found that the link is not available anymore
            7. Only use links that point directly to specific product pages, never to search results pages.
            8. Use the following categories: "Top", "Bottom", "Outer", "Footwear", "Accessories", "Bag", "Dress", "Jumpsuit", "Swimwear", "Lingerie", "Sleepwear", "Activewear".
            9. Adjust to gender in request input

            IMPORTANT OUTPUT FORMATTING:
            - Return ONLY the JSON object, without any additional text, explanation, or markdown formatting
            - Do not include code blocks, backticks, or any other non-JSON content

            Generate a JSON object following this exact structure:
            {
                "products": [
                    {
                        "category": "XXX",
                        "name": "<specific product name>",
                        "links": [
                            "<direct product link - NOT search results>",
                            "<direct product link - NOT search results>"
                        ]
                    },
                    {
                        "category": "XXX",
                        "name": "<specific product name>",
                        "links": [
                            "<direct product link - NOT search results>",
                            "<direct product link - NOT search results>"
                        ]
                    }
                ]
            }

            DO NOT MAKE UP LINKS. Only use links that point directly to specific product pages, never to search results pages.
            Give me a few of items that match with the event or occasion or the user input.
            """

        # Initialize the LLM
        self.llm = ChatGroq(
            model="meta-llama/llama-4-maverick-17b-128e-instruct",
            temperature=0,
            seed=100,
            top_p=0.0000000002
        )

        # Create a wrapper function to ensure search results are properly formatted as strings
        def search_and_format(query):
            tavily = TavilySearchResults(
                api_key=os.getenv("TAVILY_API_KEY"),
                max_results=25,
                search_depth='basic',
                topic="fashion",
                include_domains=["shop-id.tokopedia.com"],
                exclude_domains=[
                    "https://www.tokopedia.com/find/",
                    "https://shopee.co.id/search?keyword="
                ]
            )
            results = tavily.invoke(query)
            # Ensure the results are returned as a properly formatted string
            if isinstance(results, list):
                return json.dumps(results)
            return str(results)

        # Initialize the search tool with the wrapper function
        self.search_tool = Tool(
            name="Product Search based on event",
            func=search_and_format,
            description=self.prompt_dari_text,
            max_retries=10,
        )

        # Create the agent
        self.agent_executor = create_react_agent(self.llm, [self.search_tool])

    def search(self, user_input):
        try:
            response = self.agent_executor.invoke({"messages": user_input})
            content = response['messages'][-1].content
            print(content)
            try:
                # Try to extract JSON from code blocks first
                json_match = re.search(r'```(?:json)?\s*(.*?)\s*```', content, re.DOTALL)
                if json_match:
                    content = json_match.group(1)

                # Parse the JSON to validate it
                parsed_json = json.loads(content)
                # Return formatted JSON string
                return json.dumps(parsed_json, ensure_ascii=False)
            except json.JSONDecodeError:
                # Return a valid error JSON if parsing fails
                return json.dumps({"error": "Could not extract valid JSON from response"})

        except Exception as e:
            # Handle any other exceptions during agent execution
            return json.dumps({"error": f"Agent execution error: {str(e)}"})

In [76]:
from langchain_community.tools.tavily_search import TavilySearchResults
from langgraph.prebuilt import create_react_agent
from langchain_groq import ChatGroq
from langchain.tools import Tool
from dotenv import load_dotenv
import os
import json
import re

load_dotenv()

class FashionSearchAgentforimage:
    def __init__(self):
        # Load environment variables
        self.prompt_dari_image = """This tool is used for searching fashion products based on image caption descriptions. Your task is to search for real, available product links that match the outfit described in the image caption input.

            IMPORTANT SEARCH INSTRUCTIONS:
            1. Parse the image caption to identify specific clothing items, their colors, styles, and details
            3. YOU MUST RETURN DIRECT PRODUCT LINKS ONLY. Valid links should point to specific products
            3. Click through to actual product pages to get the direct product URLs
            5. If you cannot find a suitable item after searching, remove that category from the JSON object. Do not include empty categories
            6. You need to make sure the link is available, as links can expire or products can go out of stock
            7. Use the following categories based on what's described in the image: "Hijab/Headwear", "Top", "Bottom", "Dress", "Jumpsuit", "Outer", "Footwear", "Bag/Purse", "Accessories", "Swimwear", "Lingerie", "Sleepwear", "Activewear"
            8. Adjust to the gender and style as described in the image caption

            IMPORTANT OUTPUT FORMATTING:
            - Return ONLY the JSON object, without any additional text, explanation, or markdown formatting
            - Do not include code blocks, backticks, or any other non-JSON content

            Generate a JSON object following this exact structure:
            {
                "products": [
                    {
                        "category": "XXX",
                        "name": "<specific product name>",
                        "links": [
                            "<direct product link - NOT search results>",
                            "<direct product link - NOT search results>"
                        ]
                    },
                    {
                        "category": "XXX",
                        "name": "<specific product name>",
                        "links": [
                            "<direct product link - NOT search results>",
                            "<direct product link - NOT search results>"
                        ]
                    }
                ]
            }

            DO NOT MAKE UP LINKS. Only use links that point directly to specific product pages, never to search results pages."""
        # Initialize the LLM
        self.llm = ChatGroq(
            model="meta-llama/llama-4-maverick-17b-128e-instruct",
            temperature=0,
            seed=100,
            top_p=0.0000000002
        )

        # Create a wrapper function to ensure search results are properly formatted as strings
        def search_and_format(query):
            tavily = TavilySearchResults(
                api_key=os.getenv("TAVILY_API_KEY"),
                max_results=25,
                search_depth='basic',
                topic="fashion",
                include_domains=["shop-id.tokopedia.com"],
                exclude_domains=[
                    "https://www.tokopedia.com/find/",
                    "https://shopee.co.id/search?keyword="
                ]
            )
            results = tavily.invoke(query)
            # Ensure the results are returned as a properly formatted string
            if isinstance(results, list):
                return json.dumps(results)
            return str(results)

        # Initialize the search tool with the wrapper function
        self.search_tool = Tool(
            name="Product Search based on event",
            func=search_and_format,
            description=self.prompt_dari_image,
            max_retries=10,
        )

        # Create the agent
        self.agent_executor = create_react_agent(self.llm, [self.search_tool])

    def search(self, user_input):
        try:
            response = self.agent_executor.invoke({"messages": user_input})
            content = response['messages'][-1].content
            print(content)
            try:
                # Try to extract JSON from code blocks first
                json_match = re.search(r'```(?:json)?\s*(.*?)\s*```', content, re.DOTALL)
                if json_match:
                    content = json_match.group(1)

                # Parse the JSON to validate it
                parsed_json = json.loads(content)
                # Return formatted JSON string
                return json.dumps(parsed_json, ensure_ascii=False)
            except json.JSONDecodeError:
                # Return a valid error JSON if parsing fails
                return json.dumps({"error": "Could not extract valid JSON from response"})

        except Exception as e:
            # Handle any other exceptions during agent execution
            return json.dumps({"error": f"Agent execution error: {str(e)}"})

In [77]:
import base64
import os
import requests
from io import BytesIO
from dotenv import load_dotenv
from groq import Groq

load_dotenv()

class OutfitDescriber:
    def __init__(self, model="meta-llama/llama-4-scout-17b-16e-instruct", temperature=0, top_p=0.002, seed=42):

        self.client = Groq()
        self.model = model
        self.temperature = temperature
        self.top_p = top_p
        self.seed = seed

    def encode_image(self, image_path):
        # Check if image_path is a URL or a local file path
        if image_path.startswith(('http://', 'https://')):
            # Handle URL
            response = requests.get(image_path)
            if response.status_code == 200:
                return base64.b64encode(response.content).decode('utf-8')
            else:
                raise Exception(f"Failed to fetch image from URL: {response.status_code}")
        else:
            # Handle local file path
            with open(image_path, "rb") as image_file:
                return base64.b64encode(image_file.read()).decode('utf-8')

    def describe_outfit(self, image_path: str) -> str:
        b64_image = self.encode_image(image_path)

        response = self.client.chat.completions.create(
            model=self.model,
            messages=[
                {
                    "role": "user",
                    "content": [
                        {
                            "type": "text",
                            "text": ("""
                                - Describe the outfit components in the image, and the gender of the person.
                                - Provide the description in a single short paragraph, without using lists or headings.
                                - Include details about: Top, Bottom, Footwear, Head Accessories, Accessories, Bags
                                - If the item is not present, do not include or mention it in the description.
                                - Provide the basic color each item without pattern, and the brand if possible.
                                - Do not talk about missing items or overall color palette.
                                - Do not include any other information or context about the image.
                                - DO NOT PROVIDE A JSON OBJECT OR ANY OTHER FORMATTING.
                                - DO NOT INCLUDE EXPLANATIONS LIKE "She is not wearing" or "He is not wearing."""
                            ),
                        },
                        {
                            "type": "image_url",
                            "image_url": {
                                "url": f"data:image/jpeg;base64,{b64_image}",
                            }
                        }
                    ]
                }
            ],
            temperature=self.temperature,
            top_p=self.top_p,
            stream=False,
            seed=self.seed,
        )

        print('image desc',response.choices[0].message.content)
        return response.choices[0].message.content

In [78]:
import json

class FashionAppAgent:
    def __init__(self):
        self.fashion_search_agent = FashionSearchAgent()
        self.outfit_describer = OutfitDescriber()
        self.FashionSearchAgentforimage = FashionSearchAgentforimage()

    def check_input_image_or_event(self, user_input):
        if user_input.lower().endswith(('.jpg', '.jpeg', '.png', '.webp')):
            return "image"
        else:
            return "event"

    def main(self, user_input):
        input_type = self.check_input_image_or_event(user_input)
        outfit_description = None  # Inisialisasi awal
        try:
            if input_type == "image":
                # First get description from image
                outfit_description = self.outfit_describer.describe_outfit(user_input)
                # Then search for products based on that description
                result = self.FashionSearchAgentforimage.search(outfit_description)
            else:
                outfit_description = user_input  # anggap input teks adalah deskripsi
                result = self.fashion_search_agent.search(user_input)

            return json.loads(result)

        except Exception as e:
            print(f"Error: {e}")
            return e

Agent = FashionAppAgent()


c:\Users\wanda\AppData\Local\Programs\Python\Python39\lib\site-packages\langchain_groq\chat_models.py:364: UserWarning: WARNING! seed is not default parameter.
                    seed was transferred to model_kwargs.
                    Please confirm that seed is what you intended.
  warnings.warn(
c:\Users\wanda\AppData\Local\Programs\Python\Python39\lib\site-packages\langchain_groq\chat_models.py:364: UserWarning: WARNING! top_p is not default parameter.
                    top_p was transferred to model_kwargs.
                    Please confirm that top_p is what you intended.
  warnings.warn(
c:\Users\wanda\AppData\Local\Programs\Python\Python39\lib\site-packages\langchain_groq\chat_models.py:364: UserWarning: WARNING! seed is not default parameter.
                    seed was transferred to model_kwargs.
                    Please confirm that seed is what you intended.
  warnings.warn(
c:\Users\wanda\AppData\Local\Programs\Python\Python39\lib\site-packages\langchain_groq\chat

In [80]:
# Example usage

input = r"https://res.cloudinary.com/daeebexlo/image/upload/v1746435517/ychdxmx2lnvgumrbfava.webp"
# input = "Saya cewe mau malam pertama"
result = Agent.main(input)
print('\n\n',type(result))
print(result)

Error: Error code: 503 - {'error': {'message': 'Service Unavailable', 'type': 'internal_server_error'}}


 <class 'groq.InternalServerError'>
Error code: 503 - {'error': {'message': 'Service Unavailable', 'type': 'internal_server_error'}}
